# CMPE 256 — Song Recommender System
## Starter Notebook

This notebook gives you everything you need to **get started** with the Kaggle competition:
- How to load the data files
- How to build and validate a prediction
- How to generate, download and submit a valid submission file

The model used here is the **Global Mean baseline**, the simplest possible predictor.  
Your job is to beat it.

---
> **Note:** EDA (Exploratory Data Analysis) is intentionally not included here. You are encouraged to explore the data yourself, understanding it is part of the assignment.

In [14]:
import os
import pandas as pd
from dotenv import load_dotenv

# Load KAGGLE_USERNAME and KAGGLE_KEY from the .env file in this directory
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath("RS_HW2.ipynb")), ".env"))

DATA_DIR = os.path.abspath("/home/anthony/repos/cmpe256/data/raw") + "/" 
# os.path.join(os.path.dirname(os.path.abspath("/home/anthony/repos/cmpe256/data/raw")), "") + "/"
os.makedirs(DATA_DIR, exist_ok=True)

!pip install kaggle python-dotenv -q
!kaggle competitions download -c cmpe-256-song-recommender-system33 -p {DATA_DIR}
!unzip -q -o {DATA_DIR}/*.zip -d {DATA_DIR}


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


## Step 1 — Install & Import Libraries

In [25]:
# !pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from collections import defaultdict
from collections.abc import Callable 

print('Libraries ready.')

Libraries ready.


## Step 2 — Load the Data

Four files are provided:

| File | Description |
|------|-------------|
| `train.csv` | User-song pairs with ratings (1.0–10.0). Use this to train your model. |
| `test.csv` | User-song pairs **without** ratings. You must predict these. |
| `sample_submission.csv` | Shows the exact format your submission must follow. |
| `song_data.csv` | Optional song metadata (title, artist, album, year). Useful for hybrid models. |

In [15]:
print(f'Using data directory: {DATA_DIR}')
train      = pd.read_csv(DATA_DIR + 'train.csv',             encoding='latin-1')
test       = pd.read_csv(DATA_DIR + 'test.csv',              encoding='latin-1')
sample_sub = pd.read_csv(DATA_DIR + 'sample_submission.csv', encoding='latin-1')
song_data  = pd.read_csv(DATA_DIR + 'song_data.csv',         encoding='latin-1')

print('=== train.csv ===')
print(f'  Rows   : {len(train):,}')
print(f'  Columns: {list(train.columns)}')
display(train.head(3))

print('\n=== test.csv ===')
print(f'  Rows   : {len(test):,}')
print(f'  Columns: {list(test.columns)}')
display(test.head(3))

print('\n=== sample_submission.csv ===')
print(f'  Rows   : {len(sample_sub):,}')
print(f'  Columns: {list(sample_sub.columns)}')
display(sample_sub.head(3))

print('\n=== song_data.csv ===')
print(f'  Rows   : {len(song_data):,}')
print(f'  Columns: {list(song_data.columns)}')
display(song_data.head(3))

Using data directory: /home/anthony/repos/cmpe256/data/raw/
=== train.csv ===
  Rows   : 4,254,583
  Columns: ['user_id', 'song_id', 'rating']


,user_id,song_id,rating
0,1619409,1003008,10.00
1,1006031,1006608,5.50
2,1376103,1001728,4.25



=== test.csv ===
  Rows   : 1,063,646
  Columns: ['user_id', 'song_id']


,user_id,song_id
0,1553719,1000121
1,1089653,1001182
2,1536350,1008101



=== sample_submission.csv ===
  Rows   : 1,063,646
  Columns: ['RecommendationId', 'rating']


,RecommendationId,rating
0,1553719-1000121,5.37
1,1089653-1001182,5.37
2,1536350-1008101,5.37



=== song_data.csv ===
  Rows   : 198,724
  Columns: ['song_id', 'title', 'release', 'artist_name', 'year']


,song_id,title,release,artist_name,year
0,1191338,Silent Night,Monster Ballads X-Mas,Faster Pussy cat,2003
1,1178322,L'antarctique,Des cobras des tarentules,3 Gars Su'l Sofa,2007
2,1052052,Ethos of Coercion,Descend Into Depravity,Dying Fetus,2009


## Step 3 — Build a Validation Split

Before submitting to Kaggle, always validate your model **locally** using a held-out portion of `train.csv`.  
This gives you a reliable RMSE estimate without wasting Kaggle submissions.

We use an 80/20 split here.

In [17]:
df_train, df_val = train_test_split(train, test_size=0.2, random_state=42)

print(f'Training rows  : {len(df_train):,}')
print(f'Validation rows: {len(df_val):,}')

def rmse(y_true, y_pred):
    """Compute Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

Training rows  : 3,403,666
Validation rows: 850,917


## Step 4 — Baseline Model (Global Mean)

The simplest possible model: predict the **global average rating** for every user-song pair.

This is the official competition baseline. **Your goal is to beat this RMSE.**

In [ ]:
# Compute global mean from training split only
global_mean = df_train['rating'].mean()
print(f'Global mean rating: {global_mean:.4f}')

# Predict the same value for every validation pair
val_preds = np.full(len(df_val), global_mean)
baseline_rmse = rmse(df_val['rating'], val_preds)
print(f'Baseline RMSE (validation): {baseline_rmse:.4f}')
print()
print('Beat this score on the public leaderboard to earn ranking points.')

Global mean rating: 5.3703
(850917,)
Baseline RMSE (validation): 1.7316

Beat this score on the public leaderboard to earn ranking points.


## Step 5 — Train on Full Data & Generate Predictions

Once you are happy with your model's validation RMSE, retrain it on the **full** `train.csv` (not just the 80% split) before generating your submission. This uses all available signal.

In [19]:
# Retrain on the full training set
global_mean_full = train['rating'].mean()

# Predict for every row in test.csv
# Replace this line with your own model's predictions
test_preds = np.full(len(test), global_mean_full)

print(f'Predictions shape : {test_preds.shape}')
print(f'Prediction range  : [{test_preds.min():.3f}, {test_preds.max():.3f}]')
print(f'Prediction mean   : {test_preds.mean():.4f}')

Predictions shape : (1063646,)
Prediction range  : [5.370, 5.370]
Prediction mean   : 5.3703


## Matrix Factorization with Stochatic Gradient Descent

In [ ]:
# Core of Matrix Factorization
# R (m x n)
# ^r_ui = = p_u * (q_i)^T
# p_u -> P (m x k)
# q_i -> Q (n x k)
# Goal R = P * Q^T
# E = r_ui - ^r_ui
# Objective function SSE: sum(E)
# Find Q and P where SSE is minimized
class MatrixFactorization():
    # Initializes P an Q from R.
    def __init__(self):
        pass
    # Performs Stochastic Gradient Descent to change P and Q.
    def train(self):
        pass
    # Performs dot product on P & Q to obtain r_ui
    def predict(self):
        pass


## Step 6 — Build & Validate the Submission File

The submission format requires:
- Column `RecommendationId` — formatted as `user_id-song_id`
- Column `rating` — your predicted rating, clipped to [1.0, 10.0]

We validate the format against `sample_submission.csv` before saving.

In [20]:
# ── Build submission dataframe ────────────────────────────────────────────────
submission = pd.DataFrame({
    'RecommendationId': test['user_id'].astype(str) + '-' + test['song_id'].astype(str),
    'rating': np.clip(test_preds, 1.0, 10.0)   # always clip to valid range
})

# ── Format validation ─────────────────────────────────────────────────────────
print('=== Submission Validation ===')

# 1. Column names match
assert list(submission.columns) == list(sample_sub.columns), \
    f'Column mismatch: got {list(submission.columns)}, expected {list(sample_sub.columns)}'
print('✓ Column names match sample_submission.csv')

# 2. Row count matches test.csv
assert len(submission) == len(test), \
    f'Row count mismatch: got {len(submission)}, expected {len(test)}'
print(f'✓ Row count matches test.csv ({len(submission):,} rows)')

# 3. No missing values
assert submission.isnull().sum().sum() == 0, 'Submission contains NaN values!'
print('✓ No missing values')

# 4. Ratings within valid range
assert submission['rating'].between(1.0, 10.0).all(), 'Some ratings are outside [1.0, 10.0]!'
print('✓ All ratings within valid range [1.0, 10.0]')

# 5. RecommendationId format looks correct
id_check = submission['RecommendationId'].str.match(r'^\d+-\d+$').all()
assert id_check, 'Some RecommendationId values have unexpected format!'
print('✓ RecommendationId format is correct (user_id-song_id)')

print()
print('All checks passed. Ready to submit!')
print()
display(submission.head(5))

=== Submission Validation ===
✓ Column names match sample_submission.csv
✓ Row count matches test.csv (1,063,646 rows)
✓ No missing values
✓ All ratings within valid range [1.0, 10.0]
✓ RecommendationId format is correct (user_id-song_id)

All checks passed. Ready to submit!



,RecommendationId,rating
0,1553719-1000121,5.370284
1,1089653-1001182,5.370284
2,1536350-1008101,5.370284
3,1619708-1017052,5.370284
4,1048204-1013148,5.370284


## Step 7 — Save & Download the Submission File

In [21]:
OUTPUT_FILE = DATA_DIR + 'solution.csv'
submission.to_csv(OUTPUT_FILE, index=False)
print(f'Saved to: {OUTPUT_FILE}')

Saved to: /home/anthony/repos/cmpe256/data/raw/solution.csv


## Step 8 — Check if solution file format matching with sample solution file

In [22]:
sub    = pd.read_csv(OUTPUT_FILE)
sample = pd.read_csv(DATA_DIR + 'sample_submission.csv')

print('=== Your Submission ===')
print('Columns:', sub.columns.tolist())
print('Rows:   ', len(sub))
print('dtypes:')
print(sub.dtypes)
print()
print('Preview:')
print(sub.head(3))

print()
print('=== Sample Submission ===')
print('Columns:', sample.columns.tolist())
print('Rows:   ', len(sample))
print('dtypes:')
print(sample.dtypes)
print()
print('Preview:')
print(sample.head(3))

print()
print('=== Validation Checks ===')
print('Column names match:       ', sub.columns.tolist() == sample.columns.tolist())
print('Row counts match:         ', len(sub) == len(sample))
print('No NaN in rating:         ', sub[sub.columns[1]].isna().sum() == 0)
print('Ratings in range [1, 10]: ', sub[sub.columns[1]].between(1, 10).all())
print('IDs match sample:         ', set(sub[sub.columns[0]]) == set(sample[sample.columns[0]]))
print('ID format correct:        ', sub[sub.columns[0]].str.contains(r'^\d+-\d+$').all())

=== Your Submission ===
Columns: ['RecommendationId', 'rating']
Rows:    1063646
dtypes:
RecommendationId        str
rating              float64
dtype: object

Preview:
  RecommendationId    rating
0  1553719-1000121  5.370284
1  1089653-1001182  5.370284
2  1536350-1008101  5.370284

=== Sample Submission ===
Columns: ['RecommendationId', 'rating']
Rows:    1063646
dtypes:
RecommendationId        str
rating              float64
dtype: object

Preview:
  RecommendationId  rating
0  1553719-1000121    5.37
1  1089653-1001182    5.37
2  1536350-1008101    5.37

=== Validation Checks ===
Column names match:        True
Row counts match:          True
No NaN in rating:          True
Ratings in range [1, 10]:  True
IDs match sample:          True
ID format correct:         True


## Step 9 — Submit to kaggle directly from google colab (or upload solution csv manually)

In [23]:
!kaggle competitions submit \
    -c cmpe-256-song-recommender-system33 \
    -f {OUTPUT_FILE} \
    -m "baseline kaggle competition submission"

print('Submitted - check leaderboard at:')
print('https://www.kaggle.com/competitions/cmpe-256-song-recommender-system33/leaderboard')

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload
Submitted - check leaderboard at:
https://www.kaggle.com/competitions/cmpe-256-song-recommender-system33/leaderboard
